In [2]:
import os
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

corpus_documents = [
    "장학금 수혜 대상은 직전 학기 평점 3.5 이상이며 F학점이 없는 학생입니다.",
    "도서관 이용 시간은 평일 오전 9시부터 오후 10시까지 운영됩니다.",
    "수강신청 변경 기간에는 최대 3학점까지 추가로 정정할 수 있습니다."
]

user_question = "성적 장학금을 받으려면 평점이 어떻게 되나요?"

query_vector = embeddings_model.embed_query(user_question)
doc_vector = embeddings_model.embed_documents(corpus_documents)

def calculate_cosine_similarity(v1,v2):
    return np.dot(v1,v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def calculate_euclidean_distance(v1,v2):
    return np.linalg.norm(np.array(v1) - np.array(v2))

def calculate_dot_product(v1,v2):
    return np.dot(v1,v2)

cosine_scores = [calculate_cosine_similarity(query_vector, doc_vec) for doc_vec in doc_vector]
euclidean_distance = [calculate_euclidean_distance(query_vector, doc_vec) for doc_vec in doc_vector]
dot_product = [calculate_dot_product(query_vector, doc_vec) for doc_vec in doc_vector]

print("=== [문서별 유사도 지표 비교] ===")
for i in range(len(corpus_documents)):
    print(f"[문서 {i +1}]")
    print(f" - 코사인 유사도: {cosine_scores[i]:.4f} (높을수록 일치)")
    print(f" - 유클리디안 거리: {euclidean_distance[i]:.4f} (낮을수록 일치)")
    print(f" - 내적 결과: {dot_product[i]:.4f}")

best_index = int(np.argmax(cosine_scores))
selected_context = corpus_documents[best_index]
print(f"\n최종 검색된 최적 문맥: {selected_context}")


=== [문서별 유사도 지표 비교] ===
[문서 1]
 - 코사인 유사도: 0.4146 (높을수록 일치)
 - 유클리디안 거리: 1.0818 (낮을수록 일치)
 - 내적 결과: 0.4145
[문서 2]
 - 코사인 유사도: 0.0797 (높을수록 일치)
 - 유클리디안 거리: 1.3563 (낮을수록 일치)
 - 내적 결과: 0.0797
[문서 3]
 - 코사인 유사도: 0.2372 (높을수록 일치)
 - 유클리디안 거리: 1.2350 (낮을수록 일치)
 - 내적 결과: 0.2372

최종 검색된 최적 문맥: 장학금 수혜 대상은 직전 학기 평점 3.5 이상이며 F학점이 없는 학생입니다.


In [3]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = init_chat_model(
    model_provider="openai",
    model="gpt-6-luna",
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "제공된 [참고 문서]의 내용에만 근거하여 정확하게 답변하세요.\n\n[참고 문서]:\n{context}"),
    ("user", "{question}")
])

rag_chain = prompt | model | StrOutputParser()

response = rag_chain.invoke({
    "context": selected_context,
    "question": user_question
})

print("===[생성 답변]===")
print(response)

===[생성 답변]===
직전 학기 평점이 3.5 이상이어야 합니다. F학점도 없어야 합니다.
